In [ ]:
%config Completer.use_jedi = False
import torch
import torchvision
import torch.nn.functional as F
from torch import nn, optim
from torch.autograd import Variable
from torchvision import transforms, datasets
import os
import numpy as np
import matplotlib.pyplot as plt
import glob
from torch.utils.data import Dataset, random_split
import sys, os
from tqdm import tqdm
import re


In [ ]:
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]= "0"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print('device:',device)
print('number of devices:',torch.cuda.device_count())
print('device name:',torch.cuda.get_device_name(0))
print('current device:',torch.cuda.current_device())


In [ ]:
epochs = 100
batch_size = 64

n1 = 128
n2 = 128

depth = 8
filters = 32
inch = 1
outch = 1

validation_split = 0.111

# === Training data directories ===
# data1_dir: stack-N2N input patches (with random + linear-coupling + correlated multi-channel noise)
# data2_dir: linear-coupling noise patches (separated from data1)
# data3_dir: stack-N2N input patches without linear-coupling noise
data1_dir = './data/train/data1/'
data2_dir = './data/train/data2/'
data3_dir = './data/train/data3/'

data_dir = [data1_dir, data2_dir, data3_dir]

# === Test data directories (same structure as training data) ===
test1_dir = './data/test/data1/'
test2_dir = './data/test/data2/'
test3_dir = './data/test/data3/'
testin_dir = [test1_dir, test2_dir, test3_dir]

load_model = 'last'  # set 'last' or 'best'
save_dir = './models/model_first/'
test_dir_r = './test_out_r'
test_dir_l = './test_out_l'
test_dir_d = './test_out_d'

os.system("mkdir -p %s" %test_dir_r)
os.system("mkdir -p %s" %test_dir_l)
os.system("mkdir -p %s" %test_dir_d)
os.system("mkdir -p %s" %save_dir)


In [ ]:
def input_files(file_name):
    fin = open(file_name,"rb")
    patch = np.fromfile(fin,dtype='float32')
    fin.close()
    return patch

In [ ]:
class DasDataset(Dataset):
    def __init__(self, data_dir, n1, n2, sample_ratio=1, preload=True, transform_data1=None, transform_label=None, transform_data2=None, transform_data3=None):

        self.n1 = n1
        self.n2 = n2
        self.preload = preload
        self.sample_ratio = sample_ratio

        self.transform_data1 = transform_data1
        self.transform_label = transform_label
        self.transform_data2 = transform_data2
        self.transform_data3 = transform_data3


        data1_list = glob.glob(data_dir[0]+'patch01*')
        label_list = glob.glob(data_dir[0]+'patch02*')
        data2_list = glob.glob(data_dir[1]+'noise*')
        data3_list = glob.glob(data_dir[2]+'patch01*')  # input patch without linear-coupling noise

        self.data1_list = sorted(data1_list)
        self.label_list = sorted(label_list)
        self.data2_list = sorted(data2_list)
        self.data3_list = sorted(data3_list)

        if len(data1_list) != len(label_list):
            print("ERROR: The number of shot and mask file is different")
            sys.exit(1)

        if preload:
            ntrain = 0
            x_train = []
            y_train = []
            z_train = []
            s_train = []

            for ii in tqdm(range(len(self.data1_list))):
                patch1 = input_files(self.data1_list[ii])
                patch1 = list(patch1)

                patch2 = input_files(self.label_list[ii])
                patch2 = list(patch2)

                patch3 = input_files(self.data2_list[ii])
                patch3 = list(patch3)

                patch4 = input_files(self.data3_list[ii])
                patch4 = list(patch4)

                x_train.append(patch1)
                y_train.append(patch2)
                z_train.append(patch3)
                s_train.append(patch4)

                ntrain = ntrain + 1

                del patch1
                del patch2
                del patch3
                del patch4

            x_train = np.array(x_train)
            y_train = np.array(y_train)
            z_train = np.array(z_train)
            s_train = np.array(s_train)

            self.x_train = x_train.reshape(-1,1,n2,n1)
            self.y_train = y_train.reshape(-1,1,n2,n1)
            self.z_train = z_train.reshape(-1,1,n2,n1)
            self.s_train = s_train.reshape(-1,1,n2,n1)

            print("shape of train data:", self.x_train.shape,self.y_train.shape,self.z_train.shape,self.s_train.shape)
            print("ndata: %d" %ntrain)


    def __len__(self):
        return len(self.data1_list)


    def __getitem__(self, idx):
        if self.preload:
            data1 = self.x_train[idx]
            label = self.y_train[idx]
            data2 = self.z_train[idx]
            data3 = self.s_train[idx]
        else:
            data1 = input_files(self.data1_list[idx])
            label = input_files(self.label_list[idx])
            data2 = input_files(self.data2_list[idx])
            data3 = input_files(self.data3_list[idx])
            data1 = data1.reshape(1,self.n2,self.n1)
            label = label.reshape(1,self.n2,self.n1)
            data2 = data2.reshape(1,self.n2,self.n1)
            data3 = data3.reshape(1,self.n2,self.n1)

        if self.transform_data1:
            data1 = self.transform_data1(data1)
        if self.transform_label:
            label = self.transform_label(label)
        if self.transform_data2:
            data2 = self.transform_data2(data2)
        if self.transform_data3:
            data3 = self.transform_data3(data3)

        return data1,label,data2,data3


In [ ]:
transform_data1 = transforms.Compose([transforms.ToTensor()])
transform_label = transforms.Compose([transforms.ToTensor()])
transform_data2 = transforms.Compose([transforms.ToTensor()])
transform_data3 = transforms.Compose([transforms.ToTensor()])

In [ ]:
dataset = DasDataset(
        data_dir = data_dir,
        n1 = n1,
        n2 = n2,
        sample_ratio = 1,
        preload = True)


dataset_size = len(dataset)
print(dataset_size)
valid_size = int(validation_split * dataset_size)
train_size = dataset_size - valid_size

# Randomly split dataset into train and validation subsets
trainset, validset = random_split(dataset, [train_size, valid_size])


train_loader = torch.utils.data.DataLoader(
        dataset = trainset,
        batch_size = batch_size,
        num_workers = 10,
        shuffle = True,
        drop_last=True)

valid_loader = torch.utils.data.DataLoader(
        dataset = validset,
        batch_size = batch_size,
        num_workers = 10,
        shuffle = False,
        drop_last=False)

print('ntrain:', len(train_loader.dataset), 'nvalid:', len(valid_loader.dataset))


In [ ]:
testset = DasDataset(
        data_dir = testin_dir,
        n1 = n1,
        n2 = n2,
        sample_ratio = 1,
        preload = True)

test_loader = torch.utils.data.DataLoader(
        dataset = testset,
        batch_size = batch_size,
        num_workers = 10,
        shuffle = False,
        drop_last=False)


In [ ]:
class DnCNN(nn.Module):
    def __init__(self, depth, filters, inch, outch):
        super(DnCNN, self).__init__()
        self.depth = depth
        self.filters = filters
        self.inch = inch
        self.outch = outch
        
        
        def first_block(inch,filters):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=filters, kernel_size=3, padding=1)]
            layers += [nn.ReLU()]
            
            xx = nn.Sequential(*layers)
            return xx
            
                
        def conv_block(filters, depth):
            layers = []
            for ii in range(depth-2):
                layers += [nn.Conv2d(in_channels=filters, out_channels=filters, kernel_size=3, padding=1)]
                layers += [nn.BatchNorm2d(filters)]
                layers += [nn.ReLU()]
            
            xx = nn.Sequential(*layers)
            return xx
            
        
        def last_block(outch,filters):
            layers = []
            layers += [nn.Conv2d(in_channels=filters, out_channels=outch, kernel_size=3, padding=1)]
                        
            xx = nn.Sequential(*layers)
            return xx

        
        self.first_block = first_block(inch,filters)
        self.conv_block  = conv_block(filters,depth)
        self.last_block  = last_block(outch,filters)      

        
    def forward(self,indata):
        xx = self.first_block(indata)
        xx = self.conv_block(xx)
        xx = self.last_block(xx)
        xx = torch.sub(indata,xx)
        
        return xx

In [ ]:
class Autoencoder_r(nn.Module):
    def __init__(self):
        super(Autoencoder_r, self).__init__()

        def conv_block(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.MaxPool2d(2)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def conv_block_last(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def deconv_block(inch, outch):          
            layers = []
            layers += [nn.Upsample(scale_factor=2)]
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            decode = nn.Sequential(*layers)
            return decode
        
        self.conv1 = conv_block(1,16)
        self.conv2 = conv_block(16,32)
        self.conv3 = conv_block(32,64)
        self.conv4 = conv_block(64,128)
        self.conv5 = conv_block_last(128,256)
        self.deconv1 = deconv_block(256,128)
        self.deconv2 = deconv_block(128,64)
        self.deconv3 = deconv_block(64,32)
        self.deconv4 = deconv_block(32,16)
        self.lastblock = nn.Sequential(*[nn.Conv2d(in_channels=16, out_channels=1, kernel_size=3, padding=1)])
                
    def forward(self,x):
        en = self.conv1(x)
        en = self.conv2(en)
        en = self.conv3(en)
        en = self.conv4(en)
        en = self.conv5(en)
        de = self.deconv1(en)
        de = self.deconv2(de)
        de = self.deconv3(de)
        de = self.deconv4(de)
        de = self.lastblock(de)
        
        return de

In [ ]:
class Autoencoder_l(nn.Module):
    def __init__(self):
        super(Autoencoder_l, self).__init__()

        def conv_block(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.MaxPool2d(2)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def conv_block_last(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def deconv_block(inch, outch):          
            layers = []
            layers += [nn.Upsample(scale_factor=2)]
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            decode = nn.Sequential(*layers)
            return decode
        
        self.conv1 = conv_block(1,16)
        self.conv2 = conv_block(16,32)
        self.conv3 = conv_block(32,64)
        self.conv4 = conv_block(64,128)
        self.conv5 = conv_block_last(128,256)
        self.deconv1 = deconv_block(256,128)
        self.deconv2 = deconv_block(128,64)
        self.deconv3 = deconv_block(64,32)
        self.deconv4 = deconv_block(32,16)
        self.lastblock = nn.Sequential(*[nn.Conv2d(in_channels=16, out_channels=1, kernel_size=3, padding=1)])
                
    def forward(self,x):
        en = self.conv1(x)
        en = self.conv2(en)
        en = self.conv3(en)
        en = self.conv4(en)
        en = self.conv5(en)
        de = self.deconv1(en)
        de = self.deconv2(de)
        de = self.deconv3(de)
        de = self.deconv4(de)
        de = self.lastblock(de)
        xx = torch.sub(x,de)
        
        return xx

In [ ]:
class Autoencoder_d(nn.Module):
    def __init__(self):
        super(Autoencoder_d, self).__init__()

        def conv_block(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.MaxPool2d(2)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def conv_block_last(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def deconv_block(inch, outch):          
            layers = []
            layers += [nn.Upsample(scale_factor=2)]
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            decode = nn.Sequential(*layers)
            return decode
        
        self.conv1 = conv_block(1,16)
        self.conv2 = conv_block(16,32)
        self.conv3 = conv_block(32,64)
        self.conv4 = conv_block(64,128)
        self.conv5 = conv_block_last(128,256)
        self.deconv1 = deconv_block(256,128)
        self.deconv2 = deconv_block(128,64)
        self.deconv3 = deconv_block(64,32)
        self.deconv4 = deconv_block(32,16)
        self.lastblock = nn.Sequential(*[nn.Conv2d(in_channels=16, out_channels=1, kernel_size=3, padding=1)])
                
    def forward(self,x):
        en = self.conv1(x)
        en = self.conv2(en)
        en = self.conv3(en)
        en = self.conv4(en)
        en = self.conv5(en)
        de = self.deconv1(en)
        de = self.deconv2(de)
        de = self.deconv3(de)
        de = self.deconv4(de)
        de = self.lastblock(de)
        
        return de

In [ ]:
class CascadeNet(nn.Module):
    def __init__(self):
        super(CascadeNet, self).__init__()

        def conv_block(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.MaxPool2d(2)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def conv_block_last(inch, outch):
            layers = []
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            encode = nn.Sequential(*layers)
            return encode
        
        def deconv_block(inch, outch):          
            layers = []
            layers += [nn.Upsample(scale_factor=2)]
            layers += [nn.Conv2d(in_channels=inch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            layers += [nn.Conv2d(in_channels=outch, out_channels=outch, kernel_size=3, padding=1)]
            layers += [nn.BatchNorm2d(outch)]
            layers += [nn.ELU()]
            layers += [nn.Dropout2d(p=0.05)]
            
            decode = nn.Sequential(*layers)
            return decode
        
        self.conv1_r = conv_block(1,16)
        self.conv2_r = conv_block(16,32)
        self.conv3_r = conv_block(32,64)
        self.conv4_r = conv_block(64,128)
        self.conv5_r = conv_block_last(128,256)
        self.deconv1_r = deconv_block(256,128)
        self.deconv2_r = deconv_block(128,64)
        self.deconv3_r = deconv_block(64,32)
        self.deconv4_r = deconv_block(32,16)
        self.lastblock_r = nn.Sequential(*[nn.Conv2d(in_channels=16, out_channels=1, kernel_size=3, padding=1)])
        
        self.conv1_d = conv_block(1,16)
        self.conv2_d = conv_block(16,32)
        self.conv3_d  = conv_block(32,64)
        self.conv4_d = conv_block(64,128)
        self.conv5_d = conv_block_last(128,256)
        self.deconv1_d = deconv_block(256,128)
        self.deconv2_d = deconv_block(128,64)
        self.deconv3_d = deconv_block(64,32)
        self.deconv4_d = deconv_block(32,16)
        self.lastblock_d = nn.Sequential(*[nn.Conv2d(in_channels=16, out_channels=1, kernel_size=3, padding=1)])
        
    def forward(self,x):
        en1 = self.conv1_r(x)
        en1 = self.conv2_r(en1)
        en1 = self.conv3_r(en1)
        en1 = self.conv4_r(en1)
        en1 = self.conv5_r(en1)
        de1 = self.deconv1_r(en1)
        de1 = self.deconv2_r(de1)
        de1 = self.deconv3_r(de1)
        de1 = self.deconv4_r(de1)
        de1 = self.lastblock_r(de1)
        
        idx = torch.randperm(de1.shape[2])
        xx = de1[:,:,idx,:]
        
        en2 = self.conv1_d(xx)
        en2 = self.conv2_d(en2)
        en2 = self.conv3_d(en2)
        en2 = self.conv4_d(en2)
        en2 = self.conv5_d(en2)
        de2 = self.deconv1_d(en2)
        de2 = self.deconv2_d(de2)
        de2 = self.deconv3_d(de2)
        de2 = self.deconv4_d(de2)
        de2 = self.lastblock_d(de2)          
        
        return de1,de2 

In [ ]:
class pcc_loss(nn.Module):
    def __init__(self):
        super(pcc_loss, self).__init__()
        
    def forward(self, true, pred):
        xx = pred
        yy = true
        
        vx = xx - torch.mean(xx)
        vy = yy - torch.mean(yy)
        
        pcc_loss = torch.sum(vx * vy) / (torch.sqrt(torch.sum(vx ** 2)) * torch.sqrt(torch.sum(vy ** 2)))
        
        return 1-torch.absolute(pcc_loss)


In [ ]:
model_r = Autoencoder_r().to(device)
model_l = DnCNN(depth=depth, filters=filters, inch=inch, outch=outch).to(device)
model_d = Autoencoder_d().to(device)

optimizer_r = torch.optim.Adam(model_r.parameters(), lr=0.001)
optimizer_l = torch.optim.Adam(model_l.parameters(), lr=0.001)
optimizer_d = torch.optim.Adam(model_d.parameters(), lr=0.001)

criterion1 = nn.L1Loss()
criterion2 = nn.MSELoss()

criterion3 = nn.L1Loss()
criterion4 = pcc_loss()

criterion5 = nn.L1Loss()
criterion6 = nn.MSELoss()

print(model_r)
print(model_l)
print(model_d)


In [ ]:
def findLastCheckpoint(save_dir):
    file_list = glob.glob(os.path.join(save_dir,'model_r_*.pth.tar'))  # get name list of all .hdf5 files
    if file_list:
        epochs_exist = []
        for file_ in file_list:
            result = re.findall(".*model_r_(.*).pth.tar*",file_)
            epochs_exist.append(int(result[0]))
        initial_epoch=max(epochs_exist)   
    else:
        initial_epoch = 1 
    return initial_epoch


In [ ]:
def save_checkpoint(model, filename="my_checkpoint.pth.tar"):
    checkpoint = {
        "state_dict": model.state_dict(),
    }
    torch.save(checkpoint, filename)

In [ ]:
initial_epoch = findLastCheckpoint(save_dir=save_dir)
if initial_epoch > epochs:
    initial_epoch = epochs 
    
if initial_epoch > 1:
    if load_model == 'best':
        print('load best model')
        checkpoint = torch.load(os.path.join(save_dir,'best_model_r.pth.tar'))
        model_r.load_state_dict(checkpoint['state_dict'])
        
        checkpoint = torch.load(os.path.join(save_dir,'best_model_l.pth.tar'))
        model_l.load_state_dict(checkpoint['state_dict'])
        
        checkpoint = torch.load(os.path.join(save_dir,'best_model_d.pth.tar'))
        model_d.load_state_dict(checkpoint['state_dict'])
        
    elif load_model == 'last':
        print('resuming by loading epoch %04d' %initial_epoch)
        checkpoint = torch.load(os.path.join(save_dir,'model_r_%04d.pth.tar' %initial_epoch))
        model_r.load_state_dict(checkpoint['state_dict'])
        
        checkpoint = torch.load(os.path.join(save_dir,'model_l_%04d.pth.tar' %initial_epoch))
        model_l.load_state_dict(checkpoint['state_dict'])
      
        checkpoint = torch.load(os.path.join(save_dir,'model_d_%04d.pth.tar' %initial_epoch))
        model_d.load_state_dict(checkpoint['state_dict'])
      

In [ ]:
def train(model_r, model_l, model_d, train_loader, valid_loader, optimizer_r, optimizer_d, epoch):
    model_r.train()
    model_l.train()
    model_d.train()

    train_loss1 = 0
    train_loss2 = 0
    train_loss3 = 0


    w1 = 0.5
    w2 = 1-w1

    w3 = 0.4
    w4 = 1-w3

    w5 = 0.2
    w6 = 1-w5

    time1 = time.time()
    for batch_idx, data in enumerate(train_loader):
        inputs, labels, noise, separ = data
        inputs, labels, noise, separ = inputs.to(device), labels.to(device), noise.to(device), separ.to(device)

        # Train random-noise attenuation network (model_r)
        preds1 = model_r(inputs)

        loss1 = criterion1(preds1, labels)
        loss2 = criterion2(preds1, inputs)

        loss_r = w1*loss1 + w2*loss2

        optimizer_r.zero_grad(set_to_none=True)
        loss_r.backward()
        optimizer_r.step()

        # Train linear-coupling noise attenuation network (model_l)
        model_l.train()

        noise_pred = model_r(noise)
        separ_pred = model_r(separ)

        r1 = np.random.uniform(0.3,0.7)


        r2 = 1-r1

        preds1_true = separ_pred.detach()
        preds1_add = r2*preds1_true + r1*noise_pred
        preds2 = model_l(preds1_add)
        loss3 = criterion3(preds2, preds1_true*r2)
        loss4 = criterion4(preds2, preds1_true*r2)
        loss_l = w3*loss3 + w4*loss4

        if epoch > 5:
            optimizer_l.zero_grad(set_to_none=True)
            loss_l.backward()
            optimizer_l.step()

        # Estimate linear-coupling noise output (preds2_t) via trained model_l (eval mode)
        model_l.eval()
        preds2_t = model_l(preds1)

        # Shuffle along the channel axis for self-supervised CMN training
        idx1 = torch.randperm(preds2_t.shape[2])
        preds2_s1 = preds2_t[:,:,idx1,:].detach()

        # Train correlated multi-channel noise attenuation network (model_d)
        preds3 = model_d(preds2_s1)

        idx2 = torch.randperm(preds2_t.shape[2])
        preds2_s2 = preds2_t[:,:,idx2,:].detach()

        loss_d = w5*criterion5(preds3, preds2_s2) + w6*criterion6(preds3, preds2_s2)

        if epoch > 10:
            optimizer_d.zero_grad(set_to_none=True)
            loss_d.backward()
            optimizer_d.step()

        loss = loss_r + loss_l + loss_d

        time2 = time.time()
        tcost = time2 - time1
        if batch_idx % 30 == 0:
            print('Train epoch: {:4d} [{:6d}/{:6d} ({:0f}%)]\tLoss: {:.6f} = {:.6f} + {:.6f} + {:.6f} \tTime cost {:.3f}'.format(epoch, batch_idx*len(inputs), len(train_loader.dataset),100*batch_idx / len(train_loader), loss.item(), loss_r.item(), loss_l.item(), loss_d.item(), tcost))

        train_loss1 += loss_r.item()
        train_loss2 += loss_l.item()
        train_loss3 += loss_d.item()



    model_r.eval()
    model_l.eval()
    model_d.eval()
    val_loss1 = 0
    val_loss2 = 0
    val_loss3 = 0
    correct = 0

    with torch.no_grad():
        for batch_idx, data in enumerate(valid_loader):
            inputs, labels, noise, separ = data
            inputs, labels, noise, separ = inputs.to(device), labels.to(device), noise.to(device), separ.to(device)

            valids1 = model_r(inputs)

            loss1 = criterion1(valids1, labels)
            loss2 = criterion2(valids1, inputs)

            loss_r = w1*loss1 + w2*loss2

            noise_valid = model_r(noise)
            separ_valid = model_r(separ)

            # Validation: linear-coupling noise attenuation
            valids1_true = separ_valid.detach()

            r1 = np.random.uniform(0.3,0.7)


            r2 = 1-r1
            valids1_add = r2*valids1_true + r1*noise_valid
            valids2 = model_l(valids1_add)

            loss3 = criterion3(valids2, valids1_true*r2)
            loss4 = criterion4(valids2, valids1_true*r2)
            loss_l = w3*loss3 + w4*loss4

            valids2_t = model_l(valids1)

            # Shuffle along the channel axis
            idx1 = torch.randperm(valids2_t.shape[2])
            valids2_s1 = valids2_t[:,:,idx1,:].detach()

            # Validation: correlated multi-channel noise attenuation
            valids3 = model_d(valids2_s1)

            idx2 = torch.randperm(valids2_t.shape[2])
            valids2_s2 = valids2[:,:,idx2,:].detach()

            loss_d = w5*criterion5(valids3, valids2_s2) + w6*criterion6(valids3, valids2_s2)

            val_loss1 += loss_r.item()
            val_loss2 += loss_l.item()
            val_loss3 += loss_d.item()

    train_loss1 = train_loss1 / len(train_loader)
    train_loss2 = train_loss2 / len(train_loader)
    train_loss3 = train_loss3 / len(train_loader)


    val_loss1 = val_loss1 / len(valid_loader)
    val_loss2 = val_loss2 / len(valid_loader)
    val_loss3 = val_loss3 / len(valid_loader)


    print('=== End Train epoch: {:4d} \tValid Loss: {:.6f} {:.6f} {:.6f}'.format(epoch, val_loss1, val_loss2, val_loss3))



    return train_loss1, train_loss2, train_loss3, val_loss1, val_loss2, val_loss3


In [ ]:
import time
start = time.time()

logname = 'log.txt'
logout = open(os.path.join(save_dir,logname),'a')

bestname = 'best.txt'
bestout = open(os.path.join(save_dir,bestname),'a')

if initial_epoch == 1:
    best_loss1 = float('inf')
    best_loss2 = float('inf')
    best_loss3 = float('inf')
elif initial_epoch > 1:
    bestin = os.path.join(save_dir,bestname)
    best_val = np.loadtxt(bestin)
    if best_val.ndim == 1:
        last_row = best_val
    else:
        last_row = best_val[-1]

    print(last_row)
    best_loss1 = last_row[1]
    best_loss2 = last_row[2]
    best_loss3 = last_row[3]

for epoch in range(initial_epoch, epochs+1):
    train_loss1, train_loss2, train_loss3, valid_loss1, valid_loss2, valid_loss3 = train(model_r, model_l, model_d, train_loader, valid_loader, optimizer_r, optimizer_d, epoch)

    logout.write('{} {:.6f} {:.6f} {:.6f} {:.6f} {:.6f} {:.6f}\n'.format(epoch, train_loss1, train_loss2, train_loss3, valid_loss1, valid_loss2, valid_loss3))


    valid_loss = valid_loss1 + valid_loss2 + valid_loss3

    if valid_loss1 < best_loss1:
        best_loss1 = valid_loss1
        file_name1 = os.path.join(save_dir,"best_model_r.pth.tar")
        save_checkpoint(model_r, file_name1)
        print("=========================================== Valid loss_r: %.6f Save best model_r: epoch%04d" %(valid_loss1,epoch))
        bestout.write('{} {:.6f} {:.6f} {:.6f}\n'.format(epoch, best_loss1, best_loss2, best_loss3))

    if valid_loss2 < best_loss2:
        best_loss2 = valid_loss2
        file_name2 = os.path.join(save_dir,"best_model_l.pth.tar")
        save_checkpoint(model_l, file_name2)
        print("=========================================== Valid loss_l: %.6f Save best model_l: epoch%04d" %(valid_loss2,epoch))
        bestout.write('{} {:.6f} {:.6f} {:.6f}\n'.format(epoch, best_loss1, best_loss2, best_loss3))

    if valid_loss3 < best_loss3:
        best_loss3 = valid_loss3
        file_name3 = os.path.join(save_dir,"best_model_d.pth.tar")
        save_checkpoint(model_d, file_name3)
        print("=========================================== Valid loss_d: %.6f Save best model_d: epoch%04d" %(valid_loss3,epoch))
        bestout.write('{} {:.6f} {:.6f} {:.6f}\n'.format(epoch, best_loss1, best_loss2, best_loss3))

    if epoch % 10 == 0:
        file_name1 = os.path.join(save_dir,"model_r_%04d.pth.tar"%(epoch))
        save_checkpoint(model_r, file_name1)
        file_name2 = os.path.join(save_dir,"model_l_%04d.pth.tar"%(epoch))
        save_checkpoint(model_l, file_name2)
        file_name3 = os.path.join(save_dir,"model_d_%04d.pth.tar"%(epoch))
        save_checkpoint(model_d, file_name3)
    if epoch == epochs:
        file_name1 = os.path.join(save_dir,"model_r_%04d.pth.tar"%(epoch))
        save_checkpoint(model_r, file_name1)
        file_name2 = os.path.join(save_dir,"model_l_%04d.pth.tar"%(epoch))
        save_checkpoint(model_l, file_name2)
        file_name3 = os.path.join(save_dir,"model_d_%04d.pth.tar"%(epoch))
        save_checkpoint(model_d, file_name3)
        print("=========================================== Save last model: epoch%04d" %epoch)

end = time.time()
print(f"{end-start:.5f}sec")
logout.close()
bestout.close()


In [ ]:
import os
if os.path.exists(os.path.join(save_dir,'best_model_r.pth.tar')):
    checkpoint = torch.load(os.path.join(save_dir,'best_model_r.pth.tar'))
    model_r.load_state_dict(checkpoint['state_dict'])
    checkpoint = torch.load(os.path.join(save_dir,'best_model_l.pth.tar'))
    model_l.load_state_dict(checkpoint['state_dict'])
    checkpoint = torch.load(os.path.join(save_dir,'best_model_d.pth.tar'))
    model_d.load_state_dict(checkpoint['state_dict'])

In [ ]:
def show_results_r(model, data_loader, device):
    model.eval()

    dataiter = iter(data_loader)
    images, labels, noise, separ = next(dataiter)
    images = images.to(device)

    with torch.no_grad():
        preds1 = model(images)
        images = images.to('cpu')
        preds1 = preds1.to('cpu')

        inputs = images[:5].numpy()
        labels = labels[:5].numpy()
        preds = preds1[:5].detach().numpy()

    figure_width = 20
    figure_height = 20

    # Create subplots
    fig, axes = plt.subplots(nrows=4, ncols=5, figsize=(figure_width, figure_height))

    noise = inputs - preds
    # Plot subplots
    for i in range(5):
        # Row 1: input
        inimage = inputs[i,0,:,:]
        inimage = inimage.transpose(1,0)

        laimage = labels[i,0,:,:]
        laimage = laimage.transpose(1,0)

        primage = preds[i,0,:,:]
        primage = primage.transpose(1,0)

        noimage = noise[i,0,:,:]
        noimage = noimage.transpose(1,0)

        axes[0, i].imshow(inimage, cmap='gray')
        axes[0, i].set_title(f'Input - Plot {i+1}')
        axes[0, i].axis('off')

        # Row 2: label
        axes[1, i].imshow(laimage, cmap='gray')
        axes[1, i].set_title(f'Label - Plot {i+1}')
        axes[1, i].axis('off')

        # Row 3: prediction
        axes[2, i].imshow(primage, cmap='gray')
        axes[2, i].set_title(f'Pred - Plot {i+1}')
        axes[2, i].axis('off')

        # Row 4: residual noise (input - prediction)
        axes[3, i].imshow(noimage, cmap='gray')
        axes[3, i].set_title(f'Noise - Plot {i+1}')
        axes[3, i].axis('off')

        axes[0, i].set_aspect('auto')
        axes[1, i].set_aspect('auto')
        axes[2, i].set_aspect('auto')
        axes[3, i].set_aspect('auto')

    plt.tight_layout()
    plt.show()

    return preds1


In [ ]:
def show_results_l(model_r, model_l, data_loader, preds1, device):
    model_r.eval()
    model_l.eval()

    dataiter = iter(data_loader)
    images, labels, linear, separ = next(dataiter)
    images = images.to(device)
    linear = linear.to(device)
    separ = separ.to(device)

    with torch.no_grad():
        preds1 = model_r(separ)

        preds1_true = preds1.detach()
        preds1_add = 0.5*preds1_true + 0.5*linear
        preds2 = model_l(preds1_add)

        preds1 = preds1.to('cpu')
        linear = linear.to('cpu')
        preds2 = preds2.to('cpu')
        preds1_add = preds1_add.to('cpu')

        inputs = preds1_add[:5].numpy()
        labels = linear[:5].numpy()
        preds = preds2[:5].detach().numpy()
        noise = preds1_add[:5].detach().numpy() - preds

    figure_width = 20
    figure_height = 20

    # Create subplots
    fig, axes = plt.subplots(nrows=4, ncols=5, figsize=(figure_width, figure_height))

    # Plot subplots
    for i in range(5):
        # Row 1: input
        inimage = inputs[i,0,:,:]
        inimage = inimage.transpose(1,0)

        laimage = labels[i,0,:,:]
        laimage = laimage.transpose(1,0)

        primage = preds[i,0,:,:]
        primage = primage.transpose(1,0)

        noimage = noise[i,0,:,:]
        noimage = noimage.transpose(1,0)

        axes[0, i].imshow(inimage, cmap='gray')
        axes[0, i].set_title(f'Input - Plot {i+1}')
        axes[0, i].axis('off')

        # Row 2: target linear noise
        axes[1, i].imshow(laimage, cmap='gray')
        axes[1, i].set_title(f'Tnoise - Plot {i+1}')
        axes[1, i].axis('off')

        # Row 3: prediction
        axes[2, i].imshow(primage, cmap='gray')
        axes[2, i].set_title(f'Pred - Plot {i+1}')
        axes[2, i].axis('off')

        # Row 4: predicted noise (input - prediction)
        axes[3, i].imshow(noimage, cmap='gray')
        axes[3, i].set_title(f'Pnoise - Plot {i+1}')
        axes[3, i].axis('off')

        axes[0, i].set_aspect('auto')
        axes[1, i].set_aspect('auto')
        axes[2, i].set_aspect('auto')
        axes[3, i].set_aspect('auto')

    plt.tight_layout()
    plt.show()
    for i in range(5):
        # Row 1: input
        inimage = inputs[i,0,:,:]
        inimage = inimage.transpose(1,0)

        laimage = labels[i,0,:,:]
        laimage = laimage.transpose(1,0)

        primage = preds[i,0,:,:]
        primage = primage.transpose(1,0)

        noimage = noise[i,0,:,:]
        noimage = noimage.transpose(1,0)

        axes[0, i].imshow(inimage, cmap='gray')
        axes[0, i].set_title(f'Input - Plot {i+1}')
        axes[0, i].axis('off')

        # Row 2: label
        axes[1, i].imshow(laimage, cmap='gray')
        axes[1, i].set_title(f'Label - Plot {i+1}')
        axes[1, i].axis('off')

        # Row 3: prediction
        axes[2, i].imshow(primage, cmap='gray')
        axes[2, i].set_title(f'Pred - Plot {i+1}')
        axes[2, i].axis('off')

        # Row 4: residual noise
        axes[3, i].imshow(noimage, cmap='gray')
        axes[3, i].set_title(f'Noise - Plot {i+1}')
        axes[3, i].axis('off')

        axes[0, i].set_aspect('auto')
        axes[1, i].set_aspect('auto')
        axes[2, i].set_aspect('auto')
        axes[3, i].set_aspect('auto')

    plt.tight_layout()
    plt.show()

    return preds2


In [ ]:
def show_results_d(model, preds2, device):
    model.eval()

    preds2 = torch.Tensor(preds2)

    idx1 = torch.randperm(preds2.shape[2])
    preds2_s1 = preds2[:,:,idx1,:].detach()

    preds2_s1 = preds2_s1.to(device)

    idx2 = torch.randperm(preds2.shape[2])
    preds2_s2 = preds2[:,:,idx2,:].detach()

    with torch.no_grad():
        preds3 = model(preds2_s1)
        preds2_s1 = preds2_s1.to('cpu')

        preds3 = preds3.to('cpu')

        inputs = preds2_s1[:5].numpy()
        labels = preds2_s2[:5].numpy()
        preds = preds3[:5].detach().numpy()

    figure_width = 20
    figure_height = 20

    # Create subplots
    fig, axes = plt.subplots(nrows=4, ncols=5, figsize=(figure_width, figure_height))

    noise = inputs - preds
    # Plot subplots
    for i in range(5):
        # Row 1: input
        inimage = inputs[i,0,:,:]
        inimage = inimage.transpose(1,0)

        laimage = labels[i,0,:,:]
        laimage = laimage.transpose(1,0)

        primage = preds[i,0,:,:]
        primage = primage.transpose(1,0)

        noimage = noise[i,0,:,:]
        noimage = noimage.transpose(1,0)

        axes[0, i].imshow(inimage, cmap='gray')
        axes[0, i].set_title(f'Input - Plot {i+1}')
        axes[0, i].axis('off')

        # Row 2: target noise
        axes[1, i].imshow(laimage, cmap='gray')
        axes[1, i].set_title(f'Tnoise - Plot {i+1}')
        axes[1, i].axis('off')

        # Row 3: prediction
        axes[2, i].imshow(primage, cmap='gray')
        axes[2, i].set_title(f'Pred - Plot {i+1}')
        axes[2, i].axis('off')

        # Row 4: predicted noise
        axes[3, i].imshow(noimage, cmap='gray')
        axes[3, i].set_title(f'Pnoise - Plot {i+1}')
        axes[3, i].axis('off')

        axes[0, i].set_aspect('auto')
        axes[1, i].set_aspect('auto')
        axes[2, i].set_aspect('auto')
        axes[3, i].set_aspect('auto')

    plt.tight_layout()
    plt.show()


In [ ]:
def write_results(model, data_loader, device, test_dir):
    model.eval()

    number = 0
    with torch.no_grad():
        for batch_idx, data in enumerate(data_loader):
            inputs, labels, noise, separ = data
            inputs = inputs.to(device)

            preds = model(inputs)
            inputs = inputs.to('cpu')
            preds = preds.to('cpu')

            for jj in range(inputs.shape[0]):
                xxx = np.float32(inputs[jj,0,:,:])
                yyy = np.float32(preds[jj,0,:,:])
                zzz = np.float32(labels[jj,0,:,:])

                fout1 = open(os.path.join(test_dir,"input_%06d.bin"%number),"wb")
                fout2 = open(os.path.join(test_dir,"pred_%06d.bin"%number),"wb")
                fout3 = open(os.path.join(test_dir,"label_%06d.bin"%number),"wb")

                xxx.tofile(fout1)
                yyy.tofile(fout2)
                zzz.tofile(fout3)

                fout1.close()
                fout2.close()
                fout3.close()

                number = number + 1

    print("write_results:",number)


In [ ]:
print("=========================model_r show=========================")
preds1 = show_results_r(model_r, test_loader, device)

print("=========================model_l show=========================")
preds2 = show_results_l(model_r, model_l, test_loader, preds1, device)

print("=========================model_d show=========================")
show_results_d(model_d, preds2, device)

write_results(model_r, test_loader, device, test_dir_r)